# Caso G · 04 Agentes especialistas de calidad (mock)

> _Tutorial · Caso de uso: **G — Calidad con agentes** · Capa Medallion: **transversal** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Implementar 3 agentes-mock que actúan como evaluadores: validador de plata, auditor de MLflow, evaluador de chatbot.


## 2. Qué se aprende

- Patrón `@tool` / `@function_tool`.
- Cómo combinar herramientas en un agente.
- Cómo mockear LLM si no hay clave.


## 3. Contexto del caso de uso

Curso evalúa la calidad del proyecto con un agente que llama a herramientas. Usaremos un cliente fake.


## 4. Relación con CENTINELA+

Los agentes pueden correr como tarea diaria desde el Dashboard Adapter.


## 5. Relación con Medallion

Transversal.


## 6. Datos de entrada

Mocks.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Definimos 3 tools.


In [ ]:
def validate_silver_layer(asset_id: str = "AULA01") -> dict:
    return {
        "asset_id": asset_id, "tags_present": True, "completeness_pct": 99.2,
        "outliers": 0, "verdict": "OK",
    }

def audit_mlflow_experiment(name: str = "case_B_baseline_2026") -> dict:
    return {
        "experiment": name, "n_runs": 4, "best_MAE": 12.4, "lakefs_tag_referenced": True,
        "verdict": "OK",
    }

def evaluate_chatbot_response(question: str, expected: str) -> dict:
    score = 0.85 if expected.lower()[:6] in question.lower() else 0.4
    return {"q": question, "expected": expected, "score": score, "hallucination": score < 0.3}

print(validate_silver_layer())
print(audit_mlflow_experiment())
print(evaluate_chatbot_response("¿Cuál es la temperatura?", "valor numérico"))


## 10. Exploración paso a paso

Combinamos en un mini-agente.


In [ ]:
TOOLS = {
    "validate_silver_layer": validate_silver_layer,
    "audit_mlflow_experiment": audit_mlflow_experiment,
    "evaluate_chatbot_response": evaluate_chatbot_response,
}

def call_agent(plan: list[tuple[str, dict]]) -> list[dict]:
    out = []
    for name, kwargs in plan:
        out.append({"tool": name, "result": TOOLS[name](**kwargs)})
    return out

plan = [
    ("validate_silver_layer", {"asset_id": "AULA01"}),
    ("audit_mlflow_experiment", {"name": "case_B_baseline_2026"}),
    ("evaluate_chatbot_response", {"question": "¿Cuál fue la T media ayer?", "expected": "valor numérico"}),
]
results = call_agent(plan)
results


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Informe consolidado.


In [ ]:
report = pd.DataFrame([{**{"tool": r["tool"]}, **r["result"]} for r in results])
report


## 13. Visualizaciones explicativas

Verdict por tool.


In [ ]:
report["tool"].value_counts().plot.bar(color="#3F51B5")
plt.title("Tools invocadas")
plt.tight_layout()


## 14. Validaciones

Cada tool retorna un dict válido y JSON-serializable.


In [ ]:
import json
for r in results:
    json.dumps(r["result"])
print("Serialización OK")


## 15. Errores comunes

1. Tools que devuelven None.
2. Captura silenciosa de excepciones — el agente no detecta el fallo.
3. No registrar timing por tool.


## 16. Ejercicios propuestos

1. Reemplaza el evaluador de chatbot por embeddings simples.
2. Añade `audit_data_drift(period)`.
3. Conecta a un LLM real con `OPENAI_API_KEY`.


## 17. Cómo se reutiliza con datos reales

Las tools no cambian: solo el origen de los datos.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `08_case_H_rag_chatbot/01_arquitectura_rag_tools.ipynb`.
- Documento web del caso: `docs/use-cases/case-g-data-quality-agents.md`.
